# Progressive Sharpening Statistics

Monte-Carlo statistics of progressive sharpening (PS) for the deep-linear model, over a grid of **initialization kind** × **width** × **depth**. This notebook is a faithful Python port of the gradient-flow dynamics in `dynamics.js`.

For each cell of the grid we draw many random networks from the standard width-normalized initialization, evolve each toward a fixed norm-1 whitened target, and record these probabilities:

- **init_sharpen** — sharpness increases at the *first* gradient step
- **ever_sharpen** — sharpness ever exceeds its initial value
- **end_higher** — final sharpness exceeds initial sharpness
- **max_not_end** — the maximum sharpness is *not* at the final step
- **sharpen_then_flatten** — a model that progressively sharpens later *flattens*, i.e. its sharpness drops more than a tolerance (default **0.5%**) below its running peak after having risen above its start. The tolerance keeps finite-step-size noise from counting as flattening.

Run all cells top to bottom. The results print inline as a table and are also saved to `ps_stats.json` for the website.

In addition to these probabilities, each cell records **init_sharp** and **end_sharp** — the mean sharpness at the first and last step, averaged over the sampled networks — so the table also shows how the typical sharpness scale shifts with architecture.

## Configuration

In [1]:
WIDTHS = [2, 3, 4, 5, 6]      # input width d
DEPTHS = [1, 2, 3, 4, 5]      # number of scalar (hidden) layers n
KINDS  = ["gaussian", "uniform"]
N_SAMPLES = 1000             # random networks per (kind, d, n) cell
TARGET_NORM = 1.0            # ||p*||

# integration settings (mirror dynamics.js)
ETA = 4e-3
MAX_STEPS = 4000
LOSS_TOL = 1e-3
SEED = 20240601

# tolerance for what counts as 'flattening' after sharpening (0.5%)
FLATTEN_TOL = 0.005

import json, math, random
import numpy as np

## Dynamics (port of `dynamics.js`)

$f(x) = B\,(w_1\cdot x)$ with $B=\prod_i b_i$; whitened data with $\tfrac1N X^\top X = I$. Sharpness is the whitened Gauss–Newton top eigenvalue $\lambda_\parallel = B^2(1 + S\lVert w_1\rVert^2)$, $S=\sum_i 1/b_i^2$.

In [2]:
def random_model(rng, kind, d, n):
    """Standard width-normalized init: w1 per-coord variance 1/d, b_i unit variance."""
    if kind == "gaussian":
        s = math.sqrt(1.0 / d)
        w1 = [rng.gauss(0.0, s) for _ in range(d)]
        b  = [rng.gauss(0.0, 1.0) for _ in range(n)]
    else:  # uniform U[-a,a] with matched variance
        aw, ab = math.sqrt(3.0 / d), math.sqrt(3.0)
        w1 = [rng.uniform(-aw, aw) for _ in range(d)]
        b  = [rng.uniform(-ab, ab) for _ in range(n)]
    return w1, b

def whitened_data(d, pstar):
    """Inputs ±√d on each axis so (1/N) X^T X = I."""
    X = []
    sd = math.sqrt(d)
    for j in range(d):
        e = [0.0]*d; e[j] = sd; X.append(e)
        e2 = [0.0]*d; e2[j] = -sd; X.append(e2)
    y = [sum(pstar[i]*xk[i] for i in range(d)) for xk in X]
    return np.array(X, float), np.array(y, float)

def run_sharpness_series(w1, b, X, y):
    """Explicit-Euler gradient flow; returns the sharpness time series."""
    w1 = np.array(w1, float); b = np.array(b, float)
    N = X.shape[0]
    def sharp(w1, b):
        return float((np.prod(b)**2) * (1.0 + np.sum(1.0/b**2) * np.sum(w1**2)))
    series = [sharp(w1, b)]
    for _ in range(MAX_STEPS):
        B = np.prod(b)
        wx = X @ w1
        r = B*wx - y
        loss = 0.5 * np.mean(r*r)
        gw1 = (B / N) * (X.T @ r)
        gb = np.array([(1.0/N) * np.sum(r * wx * (B/bi)) for bi in b])
        w1 -= ETA * gw1
        b  -= ETA * gb
        series.append(sharp(w1, b))
        if loss < LOSS_TOL:
            break
    return series

## Per-run event classification

The `sharpen_then_flatten` test walks the series, tracks the running peak, and fires only once the model has risen above its start *and* later falls more than `FLATTEN_TOL` below that peak — so ordinary step-size jitter does not register as flattening.

In [3]:
def classify(series, flatten_tol=FLATTEN_TOL):
    s0, s_end, s_max = series[0], series[-1], max(series)
    i_max = series.index(s_max)
    eps = 1e-9

    # sharpen-then-flatten: rose above start, then dropped >tol below running peak
    peak = s0; sharpened = False; stf = False
    for s in series:
        if s > peak: peak = s
        if s > s0*(1+eps): sharpened = True
        if sharpened and s < peak*(1 - flatten_tol):
            stf = True
            break

    return {
        "init_sharpen":         series[1] > s0*(1+eps) if len(series) > 1 else False,
        "ever_sharpen":         s_max > s0*(1+eps),
        "end_higher":           s_end > s0*(1+eps),
        "max_not_end":          i_max < len(series)-1 and s_max > s_end*(1+eps),
        "sharpen_then_flatten": stf,
    }

METRICS = ["init_sharpen", "ever_sharpen", "end_higher", "max_not_end", "sharpen_then_flatten"]

def run_cell(kind, d, n, seed):
    rng = random.Random(seed)
    pstar = [0.0]*d; pstar[0] = TARGET_NORM
    X, y = whitened_data(d, pstar)
    counts = {m: 0 for m in METRICS}
    sum_init_sharp = 0.0   # average sharpness at t=0
    sum_final_sharp = 0.0  # average sharpness at end of training
    for _ in range(N_SAMPLES):
        w1, b = random_model(rng, kind, d, n)
        series = run_sharpness_series(w1, b, X, y)
        ev = classify(series)
        for m in METRICS:
            counts[m] += 1 if ev[m] else 0
        sum_init_sharp += series[0]
        sum_final_sharp += series[-1]
    out = {m: counts[m]/N_SAMPLES for m in METRICS}
    out["init_sharp"]  = sum_init_sharp / N_SAMPLES
    out["end_sharp"] = sum_final_sharp / N_SAMPLES
    return out

## Run the grid

With `N_SAMPLES = 1000` this is roughly an hour. To test the pipeline first, set `N_SAMPLES = 50` in the config cell and re-run.

In [4]:
results = {}
for kind in KINDS:
    results[kind] = {}
    for d in WIDTHS:
        results[kind][str(d)] = {}
        for n in DEPTHS:
            seed = SEED + hash((kind, d, n)) % 100000
            results[kind][str(d)][str(n)] = run_cell(kind, d, n, seed)
            p = results[kind][str(d)][str(n)]
            print(f"{kind:8s} d={d} n={n}: " + "  ".join(f"{m}={p[m]:.3f}" for m in METRICS))
print("\ndone")

gaussian d=2 n=1: init_sharpen=0.272  ever_sharpen=0.769  end_higher=0.769  max_not_end=0.231  sharpen_then_flatten=0.000
gaussian d=2 n=2: init_sharpen=0.353  ever_sharpen=0.823  end_higher=0.823  max_not_end=0.177  sharpen_then_flatten=0.000
gaussian d=2 n=3: init_sharpen=0.395  ever_sharpen=0.854  end_higher=0.854  max_not_end=0.146  sharpen_then_flatten=0.000
gaussian d=2 n=4: init_sharpen=0.428  ever_sharpen=0.887  end_higher=0.887  max_not_end=0.113  sharpen_then_flatten=0.000
gaussian d=2 n=5: init_sharpen=0.409  ever_sharpen=0.855  end_higher=0.855  max_not_end=0.145  sharpen_then_flatten=0.000
gaussian d=3 n=1: init_sharpen=0.217  ever_sharpen=0.723  end_higher=0.723  max_not_end=0.277  sharpen_then_flatten=0.000
gaussian d=3 n=2: init_sharpen=0.319  ever_sharpen=0.833  end_higher=0.833  max_not_end=0.167  sharpen_then_flatten=0.000
gaussian d=3 n=3: init_sharpen=0.324  ever_sharpen=0.849  end_higher=0.849  max_not_end=0.151  sharpen_then_flatten=0.000
gaussian d=3 n=4: init_s

## Inspect results inline

A quick table of the two headline metrics (probability of *initial* sharpening and of *ever* sharpening) as a width × depth grid, per init kind. Change `metric` to view any of the five.

In [5]:
def show_table(metric, kind, fmt="{:.2f}", label=None):
    print(f"{label or metric}  ({kind})")
    print("         " + "".join(f"   n={n}" for n in DEPTHS))
    for d in WIDTHS:
        row = "  ".join(fmt.format(results[kind][str(d)][str(n)][metric]) for n in DEPTHS)
        print(f"  d={d}    {row}")
    print()

# ---- headline probabilities (fraction of models, 0..1) ----
for kind in KINDS:
    show_table("init_sharpen", kind, label="P(initially sharpens)")
    show_table("ever_sharpen", kind, label="P(eventually sharpens)")

# ---- average sharpness VALUES (the Hessian top-eigenvalue magnitude) ----
# init_sharp = mean sharpness at the first step; end_sharp = mean at the last step.
for kind in KINDS:
    show_table("init_sharp", kind, fmt="{:6.2f}", label="avg initial sharpness")
    show_table("end_sharp",  kind, fmt="{:6.2f}", label="avg final sharpness")

# ---- combined init -> end view, so the change is easy to read ----
for kind in KINDS:
    print(f"avg sharpness  init -> end  ({kind})")
    print("         " + "".join(f"      n={n}" for n in DEPTHS))
    for d in WIDTHS:
        cells = []
        for n in DEPTHS:
            r = results[kind][str(d)][str(n)]
            cells.append(f"{r['init_sharp']:5.2f}->{r['end_sharp']:5.2f}")
        print(f"  d={d}   " + "  ".join(cells))
    print()

P(initially sharpens)  (gaussian)
            n=1   n=2   n=3   n=4   n=5
  d=2    0.27  0.35  0.40  0.43  0.41
  d=3    0.22  0.32  0.32  0.40  0.41
  d=4    0.16  0.26  0.32  0.35  0.38
  d=5    0.16  0.23  0.28  0.35  0.38
  d=6    0.14  0.22  0.29  0.34  0.39

P(eventually sharpens)  (gaussian)
            n=1   n=2   n=3   n=4   n=5
  d=2    0.77  0.82  0.85  0.89  0.85
  d=3    0.72  0.83  0.85  0.87  0.88
  d=4    0.74  0.82  0.85  0.84  0.85
  d=5    0.73  0.80  0.85  0.86  0.85
  d=6    0.69  0.81  0.85  0.86  0.86

P(initially sharpens)  (uniform)
            n=1   n=2   n=3   n=4   n=5
  d=2    0.21  0.30  0.31  0.36  0.38
  d=3    0.16  0.24  0.28  0.33  0.34
  d=4    0.12  0.21  0.26  0.29  0.34
  d=5    0.12  0.19  0.24  0.28  0.35
  d=6    0.12  0.18  0.26  0.28  0.34

P(eventually sharpens)  (uniform)
            n=1   n=2   n=3   n=4   n=5
  d=2    0.68  0.75  0.78  0.82  0.85
  d=3    0.63  0.74  0.78  0.83  0.81
  d=4    0.61  0.69  0.78  0.80  0.80
  d=5    0.62  0.

## Save for the website

In [6]:
out = {
    "meta": {
        "n_samples": N_SAMPLES,
        "widths": WIDTHS, "depths": DEPTHS, "kinds": KINDS,
        "target_norm": TARGET_NORM,
        "eta": ETA, "max_steps": MAX_STEPS, "loss_tol": LOSS_TOL,
        "flatten_tol": FLATTEN_TOL,
        "metrics": METRICS,
        "averages": ["init_sharp", "end_sharp"],
    },
    "results": results,
}
with open("ps_stats.json", "w") as f:
    json.dump(out, f, indent=2)
print("wrote ps_stats.json")

wrote ps_stats.json
